# Team Season - team_dash_lineups

This notebook inspects the data **downloaded** from the `team_dash_lineups` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/team_dash_lineups`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [18]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "team_dash_lineups"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [19]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: team_dash_lineups
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/team_dash_lineups
Parquet files: 30


,file,size_mb
0,team_dash_lineups__team_id=1610612737.parquet,0.062
1,team_dash_lineups__team_id=1610612738.parquet,0.062
2,team_dash_lineups__team_id=1610612739.parquet,0.062
3,team_dash_lineups__team_id=1610612740.parquet,0.061
4,team_dash_lineups__team_id=1610612741.parquet,0.061
5,team_dash_lineups__team_id=1610612742.parquet,0.061
6,team_dash_lineups__team_id=1610612743.parquet,0.062
7,team_dash_lineups__team_id=1610612744.parquet,0.061
8,team_dash_lineups__team_id=1610612745.parquet,0.063
9,team_dash_lineups__team_id=1610612746.parquet,0.062


In [20]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,team_dash_lineups__team_id=1610612737.parquet,251,64
1,team_dash_lineups__team_id=1610612738.parquet,251,64
2,team_dash_lineups__team_id=1610612739.parquet,251,64
3,team_dash_lineups__team_id=1610612740.parquet,251,64
4,team_dash_lineups__team_id=1610612741.parquet,251,64
5,team_dash_lineups__team_id=1610612742.parquet,251,64
6,team_dash_lineups__team_id=1610612743.parquet,251,64
7,team_dash_lineups__team_id=1610612744.parquet,251,64
8,team_dash_lineups__team_id=1610612745.parquet,251,64
9,team_dash_lineups__team_id=1610612746.parquet,251,64


In [21]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (7530, 64)
Total rows: 7530
Total columns: 64


In [22]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,7530,64,481920,22590,4.688


In [23]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
GROUP_VALUE,7500,99.602
TEAM_ABBREVIATION,7500,99.602
TEAM_NAME,7500,99.602
SUM_TIME_PLAYED,30,0.398
GROUP_NAME,30,0.398
GROUP_ID,30,0.398
FG3A_RANK,0,0.000
OREB_RANK,0,0.000
FT_PCT_RANK,0,0.000
FTA_RANK,0,0.000


In [24]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",0,0.0
"(0.01, 0.05]",7530,100.0
"(0.05, 0.1]",0,0.0
"(0.1, 0.25]",0,0.0
"(0.25, 0.5]",0,0.0
"(0.5, 0.75]",0,0.0
"(0.75, 1.0]",0,0.0


In [25]:
if df.empty:
    col_dist = pd.DataFrame()
else:
    col_null_pct = df.isna().mean()
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    col_bins = pd.cut(col_null_pct, bins=bins, include_lowest=True)
    col_counts = col_bins.value_counts().sort_index()
    col_dist = pd.DataFrame({
        "columns": col_counts,
        "pct_columns": (col_counts / len(col_null_pct) * 100).round(3),
    })

col_dist


,columns,pct_columns
"(-0.001, 0.01]",61,95.312
"(0.01, 0.05]",0,0.000
"(0.05, 0.1]",0,0.000
"(0.1, 0.25]",0,0.000
"(0.25, 0.5]",0,0.000
"(0.5, 0.75]",0,0.000
"(0.75, 1.0]",3,4.688


In [26]:
# Merge all teams and standardize TEAM_ABBREVIATION and TEAM_NAME by TEAM_ID, then export

files = sorted(data_dir.glob("*.parquet"))
dfs = [pd.read_parquet(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if df_all.empty:
    print("No data to merge.")
else:
    required_cols = ["TEAM_ID", "TEAM_ABBREVIATION", "TEAM_NAME"]
    missing = [c for c in required_cols if c not in df_all.columns]
    if missing:
        print(f"Missing columns: {missing}")
        print(df_all.columns)
    else:
        # Build canonical mapping per TEAM_ID using first non-null values
        team_meta = (
            df_all
            .dropna(subset=["TEAM_ID"])
            .groupby("TEAM_ID", as_index=False)
            .agg({
                "TEAM_ABBREVIATION": lambda s: s.dropna().iloc[0] if s.dropna().size else None,
                "TEAM_NAME": lambda s: s.dropna().iloc[0] if s.dropna().size else None,
            })
        )

        df_all = df_all.merge(team_meta, on="TEAM_ID", how="left", suffixes=("", "_CANON"))

        # Fill missing or inconsistent values with canonical ones
        df_all["TEAM_ABBREVIATION"] = df_all["TEAM_ABBREVIATION"].fillna(df_all["TEAM_ABBREVIATION_CANON"])
        df_all["TEAM_NAME"] = df_all["TEAM_NAME"].fillna(df_all["TEAM_NAME_CANON"])

        conflict_abbr = df_all["TEAM_ABBREVIATION"].ne(df_all["TEAM_ABBREVIATION_CANON"]) & df_all["TEAM_ABBREVIATION_CANON"].notna()
        conflict_name = df_all["TEAM_NAME"].ne(df_all["TEAM_NAME_CANON"]) & df_all["TEAM_NAME_CANON"].notna()
        df_all.loc[conflict_abbr, "TEAM_ABBREVIATION"] = df_all.loc[conflict_abbr, "TEAM_ABBREVIATION_CANON"]
        df_all.loc[conflict_name, "TEAM_NAME"] = df_all.loc[conflict_name, "TEAM_NAME_CANON"]

        df_all = df_all.drop(columns=["TEAM_ABBREVIATION_CANON", "TEAM_NAME_CANON"])

        print(df_all[["TEAM_ID", "TEAM_ABBREVIATION", "TEAM_NAME"]].drop_duplicates().head(10))

        out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "team_dash_lineups.parquet"
        df_all.to_parquet(out_path, index=False)
        print(f"Saved to: {out_path}")


         TEAM_ID TEAM_ABBREVIATION              TEAM_NAME
0     1610612737               ATL          Atlanta Hawks
251   1610612738               BOS         Boston Celtics
502   1610612739               CLE    Cleveland Cavaliers
753   1610612740               NOP   New Orleans Pelicans
1004  1610612741               CHI          Chicago Bulls
1255  1610612742               DAL       Dallas Mavericks
1506  1610612743               DEN         Denver Nuggets
1757  1610612744               GSW  Golden State Warriors
2008  1610612745               HOU        Houston Rockets
2259  1610612746               LAC            LA Clippers
Saved to: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_lineups.parquet
